# Topología Aplicada y Computacional

## Práctica 1

**Ejercicio 1:** Definir una clase para almacenar complejos simpliciales.

Antes de crear la clase `SimplicialComplex` se desarrolla la función `power_set`, que calcula el conjunto de las partes de una tupla. La implementación aprovecha la representación en base $2$ de los números para codificar cada conjunto del conjunto potencia. De esta manera, el elemento $i$-ésimo de la tupla se incluye en cada iteración si el bit $i$-ésimo del número es $1$. Recorriendo todos los números hasta $2^N - 1$, dónde $N$ es la longitud de la tupla, se obtiene el resultado.

In [2]:
from typing import Iterable

def power_set(simplicial: tuple[int, ...]) -> list[tuple[int, ...]]:
    n = len(simplicial)

    res = []

    for i in range(1, 1 << n):
        res.append(tuple(
            simplicial[j]for j in range(n) if i & (1 << j)
        ))

    return res

A continuación se implementa la clase. Para crear un objeto `SimplicialComplex` basta con proveer el iterable `generator` con los símplices iniciales, sin que haga falta añadir todas sus caras (ya que esto se hace automáticamente computando su conjunto potencia). Por otro lado, la cardinalidad se obtiene del símplice generador de mayor cardinalidad.

Se ha decidido almacenar como atributo todos los símplices del complejo simplicial, en lugar de sólo los maximales, para no tener que recomputarlos cada vez que se quiera consultar el `face_set`. De no ser así, cada llamada supondría una operación de complejidad $O(2^N)$.

In [3]:
class SimplicialComplex:
    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

En efecto, aunque el complejo simplicial sólo se defina con el triángulo $\{1,2,3\}$, las caras restantes se añaden automáticamente.

In [4]:
SimplicialComplex([(1,2,3)]).face_set

{(1,), (1, 2), (1, 2, 3), (1, 3), (2,), (2, 3), (3,)}

**Ejercicio 2:** Definir una función que devuelva la dimensión del complejo simplicial.

Como ya se guarda la cardinalidad como atributo, basta con restarle uno para obtener la dimensión.

In [5]:
class SimplicialComplex:
    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

De nuevo, se comprueba su funcionamiento.

In [6]:
SimplicialComplex([(1,2,3)]).dimension

2

**Ejercicio 3:** Definir una función que devuelva el conjunto de todas las caras del complejo simplicial.

Como en el constructor ya computa el conjunto de caras y lo guarda en el atributo `face_set`, el trabajo ya está hecho.

**Ejercicio 4:** Definir una función que devuelva el conjunto de todas las caras de dimensión dada.

Basta con filtrar las caras del atributo `face_set` con la dimensión indicada.

In [7]:
class SimplicialComplex:
    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

    def n_faces(self, dim: int) -> set[tuple[int, ...]]:
        return {face for face in self.face_set if len(face) - 1 == dim}

Comprobamos su funcionamiento.

In [8]:
SimplicialComplex([(1,2,3,4)]).n_faces(1)

{(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)}

**Ejercicio 5:** Definir una función que calcule la estrella de un símplice.

Para comenzar, se desarrolla la función auxiliar `is_face`, que indica si el símplice `s1` es cara de `s2` utilizando la definición de cara de un símplice abstracto. A partir de esta función se implementa `st` que computa la estrella del parámetro usando la definición y emite un error en caso de que el símplice provisto no sea una cara del complejo simplicial.

In [9]:
class SimplicialComplex:
    @staticmethod
    def is_face(s1: tuple[int, ...], s2: tuple[int, ...]) -> bool:
        return all(i in s2 for i in s1)

    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

    def n_faces(self, dim: int) -> set[tuple[int, ...]]:
        return {face for face in self.face_set if len(face) - 1 == dim}

    def st(self, simplicial: tuple[int, ...]) -> set[tuple[int, ...]]:
        res = {member for member in self.face_set if self.is_face(simplicial, member)}

        if not res:
            raise ValueError("'simplicial' is not a face")

        return res

Finalmente, se comprueba el funcionamiento del método.

In [10]:
SimplicialComplex([(1,2,3)]).st((1,))

{(1,), (1, 2), (1, 2, 3), (1, 3)}

**Ejercicio 6:** Definir una función que calcule el link de un símplice.

El link de un símplice $\tau$ es el conjunto de símplices cuya intersección con la estrella cerrada de $\tau$ es vacía. De este modo para implementar el método `link` es primero necesario un método para computar la estrella cerrada de un símplice. Esto es sencillo, ya que basta con usar el constructor de `SimplicialComplex` con la estrella abierta como generador, produciendo así el mínimo complejo simplicial que contiene la estrella. De este modo, una vez implementado `closed_st` basta con usar la definición de link y la función auxiliar `is_disjoint` para implementar el método `lk`.

In [11]:
class SimplicialComplex:
    @staticmethod
    def is_face(s1: tuple[int, ...], s2: tuple[int, ...]) -> bool:
        return all(i in s2 for i in s1)

    @staticmethod
    def is_disjoint(s1: tuple[int, ...], s2: tuple[int, ...]) -> bool:
        return all(i not in s1 for i in s2)

    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

    def n_faces(self, dim: int) -> set[tuple[int, ...]]:
        return {face for face in self.face_set if len(face) - 1 == dim}

    def st(self, simplicial: tuple[int, ...]) -> set[tuple[int, ...]]:
        res = {member for member in self.face_set if self.is_face(simplicial, member)}

        if not res:
            raise ValueError("'simplicial' is not a face")

        return res

    def closed_st(self, simplicial: tuple[int, ...]) -> SimplicialComplex:
        return SimplicialComplex(self.st(simplicial))

    def lk(self, simplicial: tuple[int, ...]) -> set[tuple[int, ...]]:
        closed = self.closed_st(simplicial)
        return {elem for elem in closed.face_set if self.is_disjoint(simplicial, elem)}

Se comprueba que su funcionamiento es correcto.

In [12]:
SimplicialComplex([(1,2,3)]).lk((1,))

{(2,), (2, 3), (3,)}

**Ejercicio 7:** Definir una función que calcule la característica de 
Euler

La característica de Euler, $\chi(K)$, del complejo simplicial $K$ se calcula usando una función auxiliar `_euler_characteristic`. Para calcularla, se itera sobre cada uno de los símplices del complejo y se tiene en cuenta que cada símplice $\tau \in K$ aporta $(-1)^{dim \; \tau}$. Así, si un símplice tiene dimensión par suma $1$ a $\chi(K)$ y, si es impar, resta $1$ a $\chi(K)$. Esta función se llama en el constructor de la clase y se almacena $\chi(K)$ en la clase como atributo de nombre `euler_characteristic`.

In [20]:
    @staticmethod
    def _euler_characteristic(face_set: set[tuple[int, ...]]) -> int:
        chi = 0
    
        for face in face_set:
            if (len(face) - 1) % 2 == 0:
                chi += 1
            else:
                chi -= 1
    
        return chi

Y el constructor quedaría:

In [21]:
def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
    # Resto del constructor...
    
    self.euler_characteristic = self._euler_characteristic(self.face_set)

La clase final `SimplicialComplex` es:

In [22]:
class SimplicialComplex:
    @staticmethod
    def is_face(s1: tuple[int, ...], s2: tuple[int, ...]) -> bool:
        return all(i in s2 for i in s1)

    @staticmethod
    def is_disjoint(s1: tuple[int, ...], s2: tuple[int, ...]) -> bool:
        return all(i not in s1 for i in s2)

    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)
        
        self.euler_characteristic = self._euler_characteristic(self.face_set)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

    def n_faces(self, dim: int) -> set[tuple[int, ...]]:
        return {face for face in self.face_set if len(face) - 1 == dim}

    def st(self, simplicial: tuple[int, ...]) -> set[tuple[int, ...]]:
        res = {member for member in self.face_set if self.is_face(simplicial, member)}

        if not res:
            raise ValueError("'simplicial' is not a face")

        return res

    def closed_st(self, simplicial: tuple[int, ...]) -> SimplicialComplex:
        return SimplicialComplex(self.st(simplicial))

    def lk(self, simplicial: tuple[int, ...]) -> set[tuple[int, ...]]:
        closed = self.closed_st(simplicial)
        return {elem for elem in closed.face_set if self.is_disjoint(simplicial, elem)}

    @staticmethod
    def _euler_characteristic(face_set: set[tuple[int, ...]]) -> int:
        chi = 0
    
        for face in face_set:
            if (len(face) - 1) % 2 == 0:
                chi += 1
            else:
                chi -= 1
    
        return chi

Probamos con un ejemplo

In [23]:
k = SimplicialComplex([(1, 2, 3)])
k.euler_characteristic

1

**Ejercicio 8:** Definir una función que determine el número de 
componentes conexas del espacio subyacente

Se toma el grafo $G = (V, E)$, donde $V = Vert(K)$ y $E = K^{(1)} - Vert(K)$. Es decir, $G$ es el grafo formado por los vértices y los símplices de dimensión $1$ de $K$. Cada componente conexa de $|K|$ se corresponde con una componente conexa del grafo $G$. Así, para calcular el número de componentes conexas de $|K|$ se calcula el número de componentes conexas de $G$ usando el algoritmo de búsqueda en anchura, un algoritmo ampliamente usado en teoría de grafos.

In [31]:
    def connected_components(self) -> int:
        visited = set()

        graph = self.graph()
        vertices = [v[0] for v in self.n_faces(0)]

        components = 0

        for u in vertices:
            if u in visited:
                continue

            queue = [u]
            visited.add(u)

            while queue:
                v = queue.pop()

                for neig in graph[v]:
                    if neig not in visited:
                        queue.append(neig)
                        visited.add(neig)

            components += 1

        return components

Para realizar el resto de ejercicios crearemos una nueva clase envolvente de `SimplicialCompex`, llamada `FilteredSimplicialComplex`. Esta nueva clase contendrá, a parte del complejo simplicial (del tipo `SimplicialComplex`), dos diccionarios:
* Un diccionario de nombre `timestamps`, que relaciona un flotante con el conjunto de símplices con ese flotante asignado.
* Un diccionario de nombre `time_of`, que asigna a cada símplice su flotante asignado.
Este segundo diccionario permite consultar el flotante asignado de un símplice dado en tiempo constante, en lugar de tener que buscarlo en el diccionario de `timestamps`.

La clase `FilteredSimplicialComplex` se define inicialmente como:

In [32]:
class FilteredSimplicialComplex():
    def _finish_insertion(self, additions: set[tuple[int, ...]], time: float) -> None:
        self.timestamps[time].update(additions)

        for addition in additions:
            self.time_of[addition] = time

    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.sc = SimplicialComplex(generator)

        self.timestamps = defaultdict(set)
        self.time_of = {}

        self._finish_insertion(self.sc.face_set, 0.0)

    @property
    def face_set(self) -> set[tuple[int, ...]]:
        return self.sc.face_set

**Ejercicio 9:** Añade una función a la clase de complejos simpliciales que te permita 
añadir símplices nuevos. 

Agregaremos la función pedida a la clase original `SimplicialComplex`.

In [33]:
    def insert(self, generator: Iterable[tuple[int, ...]]) -> set[tuple[int, ...]]:
        additions = set()

        for simplicial in generator:
            power_set = utils.power_set(simplicial)
            difference = set(power_set) - self.face_set

            additions.update(difference)
            self.face_set.update(difference)

            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

        self.euler_characteristic += self._euler_characteristic(additions)

        return additions

Esta función es similar al constructor de `SimplicalComples`. En la entrada `generator` se proporciona una lista de símplices, se calculan todas sus caras usando el conjunto potencia y se añaden al atributo `face_set`. Por último, se actualiza la cardinalidad y la característica de Euler. La principal diferencia con el constructor es que esta función calcula y devuelve los símplices que se han añadido nuevos. Esto es útil por dos motivos:
1. Simplifica la actualización de la característica de Euler, que se almacena como atributo. Simplemente basta con calcular la característica de Euler de los símplices nuevos y sumarla al valor actual. Es necesario que se haga con los símplices que verdaderamente hemos añadido, pues si lo hacemos con todas las caras de `generator` podríamos contar algunos símplices más de una vez.
2. Obtener los símplices que se han añadido nuevos facilita la implementación de otras funciones de la clase `FilteredSimplicialComplex`.

**Ejercicio 10:** Añade un nuevo input que permita añadir un flotante a los símplices de 
cada inserción.

Para resolver este ejercicio crearemos una función `insert` en la clase `FilteredSimplicialComplex` y usaremos la función `SimplicialComplex.insert` definida en el ejercicio anterior. 

Para resolverlo, primero creamos una réplica de la instancia del complejo simplicial y usamos la función `SimplicialComplex.insert` del ejercico anterior sobre esta réplica para añadir los símplices necesarios de `generator` y obtener los símplices nuevos. A continuación, verificamos la propiedad que tiene que tener el argumento `time`: Para cada símplice nuevo $\tau$, se tiene que si $\sigma \le \tau$ entonces $float(\sigma) \le float(\tau)$. Esto se comprueba solo con los símplices de nueva adición (calculados con `SimplicialComplex.insert`) y usando el diccionario `time_of` con cada cara de dicho símplice. Una vez realizada dicha verificación, se sobrescribe la instancia del complejo simplicial actual por la réplica modificada y se actualizan los diccionarios `timestamps` y `time_of` con los símplices de nueva adición y el tiempo `time` dado.

In [34]:
    def insert(self, generator: Iterable[tuple[int, ...]], time: float) -> None:
        for face in generator:
            if face in self.sc.face_set:
                raise ValueError("'generator' contains an already added face")

        new_sc = SimplicialComplex(self.sc.face_set)
        additions = new_sc.insert(generator)

        for addition in additions:
            for face in utils.power_set(addition):
                t = self.time_of.get(face)

                if t and t > time:
                    raise ValueError("'generator' contains a face with a bad time value")

        self.sc = new_sc
        self._finish_insertion(additions, time)

**Ejercicio 11:** Añade un nuevo atributo a la clase que almacene la lista de todos los 
símplices ordenados según el valor del flotante correspondiente teniendo 
ordenando dos que tengan el mismo flotante asociado por su dimensión.

Este ejercicio se resuelve usando una función `filtration_order` anotada con `@property`, en vez de con un atributo.

La expresión `timestamps.items()` devuelve un iterable de elementos de la forma `(time, {simplexes})`. Así, ordenando este iterable usando `sorted` obtendremos otro iterable de la misma forma pero ordenado según `time` de menor a mayor (`sorted` en Python usa el orden lexicográfico para ordenar tuplas). Se descartan los `time` usando el patrón `_, v`. Por último, como cada `v` es un iterable de símplices y cada uno de ellos lleva asociado el mismo flotante, se ordenan por longitud de tupla (dimensión $+1$).

In [35]:
    @property
    def filtration_order(self) -> list[tuple[int, ...]]:
        return [
            simp 
            for _, v in sorted(self.timestamps.items()) 
            for simp in sorted(v, key=len)
        ]

**Ejercicio 12:** Crea una función que recupere el complejo simplicial formado por todos los 
símplices cuyo flotante asociado sea menor o igual que un flotante dado.

Primero se itera sobre `timestamps.items`, que entrega valores de la forma `(flotante, simplexes)`. Después, se filtran por los que su flotante sea menor o igual al dado y se itera sobre los símplices del conjunto `simplexes`. Todo esto genera un iterable de símplices com flotante asociado menor o igual a `time`. Por último, se crea el complejo simplicial formado por estos símplices usando la clase `SimplicialComplex`.

In [36]:
    def filtration(self, time: float) -> SimplicialComplex:
        return SimplicialComplex(
            simplex 
            for k, v in self.timestamps.items() 
            if k <= time 
            for simplex in v
        )